# Cartões por jogador
## Etapa de construção de tabela gold

Agrupado dos jogadores por cartões

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window

In [0]:
df_cards= spark.read.table("apifootball.silver.cards")

In [0]:
df_agrupado = (
    df_cards.groupBy("team_id", "team_name", "player_name")
    .agg(
        sum(when(col("card_type") == "yellow card", 1).otherwise(0)).alias("yellow_cards"),
        sum(when(col("card_type") == "red card", 1).otherwise(0)).alias("red_cards"),
        count("*").alias("qt_cards")
    )
)

In [0]:
window_yellow = Window.orderBy(col("yellow_cards").desc())
window_red = Window.orderBy(col("red_cards").desc())
window_qt = Window.orderBy(col("qt_cards").desc())

df_player_cards = (
    df_agrupado
    .withColumn("yellow_card_rank", rank().over(window_yellow))
    .withColumn("red_card_rank", rank().over(window_red))
    .withColumn("card_rank", rank().over(window_qt))
    .select(
    "team_id",
    "team_name",
    "player_name",
    "yellow_cards",
    "yellow_card_rank",
    "red_cards",
    "red_card_rank",
    "qt_cards",
    "card_rank",
    expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br')    
    )
)

In [0]:
df_player_cards.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("apifootball.gold.player_cards")

In [0]:
#spark.read.table("apifootball.gold.player_cards").display()